# Predictive Maintenance: Time-Series Incident Prediction

## 1. Problem Formulation
The objective of this project is to implement a machine learning model capable of predicting an incident (system failure/threshold breach) within the next `H` time steps, based on the previous `W` time steps of time-series metrics.

* **Dataset:** Real machine temperature logs from the Numenta Anomaly Benchmark (NAB).
* **Incident Definition:** A threshold breach defined as the temperature exceeding the 90th percentile of normal operational data.
* **Window Parameters:** `W = 12` (1 hour of historical context, assuming 5-min intervals) and `H = 6` (30 minutes prediction horizon).

## 2. Data Loading and Initial Exploration
This cell imports the necessary libraries (`pandas`, `sklearn`) and loads the dataset from the provided URL. The dataset contains time-series data of machine temperature.

In [25]:
import pandas as pd
import sklearn as sk
url = "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/machine_temperature_system_failure.csv"
df = pd.read_csv(url)

## 3. Feature Engineering
I create a new feature `temp_diff` which represents the difference in temperature from the previous time step. This can help the model detect sudden changes in temperature.

In [26]:
df['temp_diff']=df['value'].diff(1)
df=df.dropna()

I then create more features based on rolling windows. I calculate the rolling mean and standard deviation of the temperature for window sizes of 3 and 6. These features help capture the trend and volatility of the temperature over different time scales.

In [27]:
df['rolling_mean_3'] = df['value'].rolling(window=3).mean()
df['rolling_mean_6'] = df['value'].rolling(window=6).mean()
df['rolling_std_3'] = df['value'].rolling(window=3).std()
df['rolling_std_6']=df['value'].rolling(window=6).std()

df = df.dropna().reset_index(drop=True)


A preview of the DataFrame with the new features.

In [28]:
df.head(20)

,timestamp,value,temp_diff,rolling_mean_3,rolling_mean_6,rolling_std_3,rolling_std_6
0,2013-12-02 21:45:00,80.269784,1.559366,79.436679,77.918465,0.785154,2.015568
1,2013-12-02 21:50:00,80.272828,0.003044,79.751010,78.807956,0.901180,1.562859
2,2013-12-02 21:55:00,80.353425,0.080597,80.298679,79.512833,0.047436,0.939920
3,2013-12-02 22:00:00,79.486523,-0.866902,80.037592,79.737136,0.478938,0.668332
4,2013-12-02 22:05:00,80.783277,1.296754,80.207742,79.979376,0.660538,0.749637
5,2013-12-02 22:10:00,79.508159,-1.275118,79.925986,80.112333,0.742514,0.512934
6,2013-12-02 22:15:00,79.302033,-0.206126,79.864489,79.951041,0.802340,0.598533
7,2013-12-02 22:20:00,80.802624,1.500591,79.870938,80.039340,0.813419,0.687906
8,2013-12-02 22:25:00,80.377789,-0.424835,80.160815,80.043401,0.773467,0.690199
9,2013-12-02 22:30:00,80.479237,0.101448,80.553217,80.208853,0.221869,0.647683


I check the maximum temperature value in the dataset.

In [29]:
df['value'].max()

np.float64(108.5105428)

## 4. Data Splitting
The data is split into training and testing sets, with 80% of the data used for training and the remaining 20% for testing. It's important to split time-series data chronologically to avoid data leakage from the future.

In [30]:
train_size = int(len(df) * 0.8)

train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

print(f'Size of train set: {len(train_df)} rows')
print(f'Size of train set: {len(test_df)} rows')


Size of train set: 18151 rows
Size of train set: 4538 rows


## 5. Incident Labeling
An 'incident' is defined as a temperature value exceeding the 90th percentile of the training data. A new binary column `is_incident` is created to serve as the target label for my classification model.

In [31]:
threshold = train_df['value'].quantile(0.9)
print(f'Threshold of train set: {threshold:.2f}')

train_df['is_incident']=(train_df['value']>threshold).astype(int)
test_df['is_incident']=(test_df['value']>threshold).astype(int)

print(train_df.head(20))

Threshold of train set: 98.46
              timestamp      value  temp_diff  rolling_mean_3  rolling_mean_6  \
0   2013-12-02 21:45:00  80.269784   1.559366       79.436679       77.918465   
1   2013-12-02 21:50:00  80.272828   0.003044       79.751010       78.807956   
2   2013-12-02 21:55:00  80.353425   0.080597       80.298679       79.512833   
3   2013-12-02 22:00:00  79.486523  -0.866902       80.037592       79.737136   
4   2013-12-02 22:05:00  80.783277   1.296754       80.207742       79.979376   
5   2013-12-02 22:10:00  79.508159  -1.275118       79.925986       80.112333   
6   2013-12-02 22:15:00  79.302033  -0.206126       79.864489       79.951041   
7   2013-12-02 22:20:00  80.802624   1.500591       79.870938       80.039340   
8   2013-12-02 22:25:00  80.377789  -0.424835       80.160815       80.043401   
9   2013-12-02 22:30:00  80.479237   0.101448       80.553217       80.208853   
10  2013-12-02 22:35:00  81.423560   0.944323       80.760196       80.315567  

## 6. Windowing
I define a function `create_sliding_windows` to transform the time-series data into a supervised learning problem. For each time point, I create a feature vector `X` consisting of the previous `W` time steps of data and a label `y` indicating if an incident occurs within the next `H` time steps.

In [32]:
import numpy as np

W = 12
H = 6

feature_columns = ['value','temp_diff', 'rolling_mean_3', 'rolling_mean_6','rolling_std_3', 'rolling_std_6']
def create_sliding_windows(data_values, labels, W, H):
    X, y = [],[]
    for i in range(len(data_values) - W - H + 1):
        window_x = data_values[i:i + W]
        window_x_flattened = window_x.flatten()
        window_y_labels = labels[i + W : i + W + H]

        label = 1 if (np.any(window_y_labels) == 1) else 0

        X.append(window_x_flattened)
        y.append(label)

    return np.array(X), np.array(y)

The `create_sliding_windows` function is applied to both the training and testing sets to generate the final datasets for model training and evaluation.

In [33]:
X_train, y_train = create_sliding_windows(train_df[feature_columns].values, train_df['is_incident'].values,W,H)
X_test, y_test = create_sliding_windows(test_df[feature_columns].values, test_df['is_incident'].values,W,H)

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'Account of incidents in y_train: {np.sum(y_train == 1)} (in {len(y_train)} all ceses)')

X_train shape: (18134, 72)
y_train shape: (18134,)
Account of incidents in y_train: 2172 (in 18134 all ceses)


## 7. Model Training
An XGBoost classifier is chosen for this task. The `scale_pos_weight` parameter is used to handle the class imbalance between incident and non-incident cases.

In [34]:
import xgboost as xgb
scale_weight = sum(y_train == 0)/ sum(y_train== 1)
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_weight,
)

model.fit(X_train, y_train)


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

## 8. Feature Importance
I inspect the feature importances from the trained XGBoost model to understand which features are most influential in predicting incidents.

In [35]:
print('Feature importances:')
for i, importance in enumerate(model.feature_importances_):
    print(f'{i+1}: {importance:.3f}')

Feature importances:
1: 0.005
2: 0.002
3: 0.003
4: 0.007
5: 0.002
6: 0.002
7: 0.002
8: 0.001
9: 0.013
10: 0.007
11: 0.002
12: 0.003
13: 0.001
14: 0.002
15: 0.007
16: 0.005
17: 0.002
18: 0.005
19: 0.001
20: 0.001
21: 0.007
22: 0.007
23: 0.002
24: 0.005
25: 0.002
26: 0.002
27: 0.002
28: 0.003
29: 0.002
30: 0.002
31: 0.004
32: 0.002
33: 0.006
34: 0.003
35: 0.001
36: 0.002
37: 0.004
38: 0.001
39: 0.002
40: 0.001
41: 0.002
42: 0.003
43: 0.003
44: 0.001
45: 0.004
46: 0.003
47: 0.003
48: 0.001
49: 0.004
50: 0.002
51: 0.002
52: 0.007
53: 0.002
54: 0.003
55: 0.003
56: 0.002
57: 0.004
58: 0.003
59: 0.001
60: 0.003
61: 0.032
62: 0.001
63: 0.002
64: 0.013
65: 0.001
66: 0.003
67: 0.025
68: 0.002
69: 0.587
70: 0.144
71: 0.002
72: 0.001


## 9. Model Evaluation
Finally, the model's performance is evaluated on the test set using various metrics like the classification report, confusion matrix, and accuracy score. This gives me an idea of how well the model generalizes to unseen data.

In [36]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print(classification_report(y_train, model.predict(X_train)))
print(classification_report(y_test, model.predict(X_test)))
print(confusion_matrix(y_test, model.predict(X_test)))
print(accuracy_score(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     15962
           1       0.97      1.00      0.99      2172

    accuracy                           1.00     18134
   macro avg       0.99      1.00      0.99     18134
weighted avg       1.00      1.00      1.00     18134

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      3318
           1       0.92      0.97      0.95      1203

    accuracy                           0.97      4521
   macro avg       0.96      0.97      0.96      4521
weighted avg       0.97      0.97      0.97      4521

[[3220   98]
 [  34 1169]]
0.9708029197080292


## 10. Analysis of the Results, Limitations and Business Adaptation

* **Business Metrics Focus:** In a real-world predictive maintenance system, Accuracy is a misleading metric due to class imbalance. The primary focus is **Recall (Sensitivity)**.
* **Performance:** The model achieved an outstanding **Recall of ~97%** on the unseen test set, successfully anticipating 1169 out of 1203 system failures.
* **Precision-Recall Tradeoff:** The model maintains a high precision (92%), meaning it generates very few False Positives (98 out of over 3300 normal instances). This effectively prevents "Alert Fatigue" for the maintenance team while ensuring critical outages are caught 30 minutes in advance.
* **Limitations:** The current model relies on a static 90th percentile threshold calculated on the training set. In a real-world scenario with hardware aging or seasonal workload changes, this threshold might need dynamic recalibration (e.g., using rolling quantiles) to prevent concept drift over months or years.

### Adaptation to a Real Alerting System
To deploy this in a production environment:
1. **Streaming Pipeline:** The model would consume a live stream of IoT sensor metrics (e.g., via Apache Kafka or AWS Kinesis).
2. **Microservice Inference:** Wrapped in a FastAPI/Docker container, engineering rolling features on the fly and inferring predictions every 5 minutes.
3. **Alerting Integration:** If the model predicts an incident (`1`), it triggers a webhook to PagerDuty/Slack to dispatch maintenance technicians before catastrophic hardware failure occurs.